# Change Data Feed (CDF) med Delta Sharing

Dette notebooken demonstrerer hvordan du bruker **Change Data Feed (CDF)** til å spore endringer i Delta-tabeller over tid.

In [7]:
! pip install -r requirements.txt

In [8]:
import os
import json
import subprocess
import delta_sharing
import delta_sharing
import pandas as pd
from google.cloud import storage

from src.auth import generate_access_token
from src.utils import (
    fetch_config_share,
    create_credentials_config,
)

In [9]:
project_id = "innsikt-data-dev-ec28"
project_num = "614733074632"
provider_full_identifier = f"projects/{project_num}/locations/global/workloadIdentityPools/skyporten-bi-dev/providers/skyporten-bi-provider-dev"
random_id = "9ivj"
schema_name = "matrikkel_silver_v1_ext"
share_name = f"{random_id}-dev"

CREDENTIALS_PATH = "credentials.json"
TOKEN_PATH = "tmp_maskinporten_token.txt"
CONFIG_PATH = "configs/config.json"
SOURCE_PATH = "share/config.share" #fil

In [10]:
create_credentials_config(provider_full_identifier, TOKEN_PATH, CREDENTIALS_PATH)

Created credential configuration file [credentials.json].


In [11]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

Generated token: eyJraWQiOiJiZFhMRVduRGpMSGpwRThPZnl5TUp4UlJLbVo3MUxCOHUxeUREbVBpdVQwIiwiYWxnIjoiUlMyNTYifQ.eyJhdWQiOiJodHRwczovL3NreXBvcnRlbi5rYXJ0dmVyay5ubyIsInN1YiI6IjAxOTI6OTcxMDQwMjM4O2thcnR2ZXJrOm1hdHJpa2tlbC5iZXJldHRpZ2V0aW50ZXJlc3NlIiwic2NvcGUiOiJrYXJ0dmVyazptYXRyaWtrZWwuYmVyZXR0aWdldGludGVyZXNzZSIsImlzcyI6Imh0dHBzOi8vdGVzdC5za3kubWFza2lucG9ydGVuLm5vIiwiY2xpZW50X2FtciI6InByaXZhdGVfa2V5X2p3dCIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJleHAiOjE3NTk5MjY0MTQsImlhdCI6MTc1OTkyNjM4NCwiY2xpZW50X2lkIjoiMWIxOWY0N2YtODBmNC00ZTM0LWE3ODMtODQ4ZWM0YjI5YTU2IiwianRpIjoiSklxSnl1Y05tTVBYVUNoX0pCLVhGdnd2MVlmOG9qQXJaU3JwZnVDZTJDMCIsImNvbnN1bWVyIjp7ImF1dGhvcml0eSI6ImlzbzY1MjMtYWN0b3JpZC11cGlzIiwiSUQiOiIwMTkyOjk3MTA0MDIzOCJ9fQ.SKeooP1FbFqtwGoQSSgU_UCurR1BVe0ajT8YhdyG4nFOAAMB8RzqPZCDDHjRahwgeBLGnrhlYsHwVPsb1RMCf-NvpXOe251_1KpbmkJqMjsKAVBLYA4TyWfJkCnErkPy-6trfK1xKz_eGtt_WiRH8mePlMsJiyzf1HI606graOaIHQHckJaH1HHy_fIPVXTdLGKaXxHnHApbTzrG-tFcdDMuvRsCl0vD6HZeJIR59hzbqMbV3N14VuIJt69ZifD_eg0hlH5oLRDV3K3DKuBsGoMwDsaQf4eIP

In [12]:
with open(CONFIG_PATH, 'r') as file:
    config = json.load(file)

token = generate_access_token(
    kid=config.get('kid'),
    scope=config.get('scope'),
    certname=config.get('certname'),
    audience=config.get('audience'),
    client_id=config.get('client_id'),
    token_url=config.get('url'),
)

s = token.get("access_token", "")

print(f"Generated token: {s}")

with open(TOKEN_PATH, 'w') as file:
    file.write(s)

Generated token: eyJraWQiOiJiZFhMRVduRGpMSGpwRThPZnl5TUp4UlJLbVo3MUxCOHUxeUREbVBpdVQwIiwiYWxnIjoiUlMyNTYifQ.eyJhdWQiOiJodHRwczovL3NreXBvcnRlbi5rYXJ0dmVyay5ubyIsInN1YiI6IjAxOTI6OTcxMDQwMjM4O2thcnR2ZXJrOm1hdHJpa2tlbC5iZXJldHRpZ2V0aW50ZXJlc3NlIiwic2NvcGUiOiJrYXJ0dmVyazptYXRyaWtrZWwuYmVyZXR0aWdldGludGVyZXNzZSIsImlzcyI6Imh0dHBzOi8vdGVzdC5za3kubWFza2lucG9ydGVuLm5vIiwiY2xpZW50X2FtciI6InByaXZhdGVfa2V5X2p3dCIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJleHAiOjE3NTk5MjY0MTUsImlhdCI6MTc1OTkyNjM4NSwiY2xpZW50X2lkIjoiMWIxOWY0N2YtODBmNC00ZTM0LWE3ODMtODQ4ZWM0YjI5YTU2IiwianRpIjoiUjBIMy1vdExuZ3pRb255U0lyc1Q3czNhd1NMV2NiaWxMbUZJMDF1TC0zdyIsImNvbnN1bWVyIjp7ImF1dGhvcml0eSI6ImlzbzY1MjMtYWN0b3JpZC11cGlzIiwiSUQiOiIwMTkyOjk3MTA0MDIzOCJ9fQ.QGwIpDWMMFOLO8fO9h1l3SXnRiUwddclCo-KsjdMQC9PeXqOTdTslAJ99f-Mr2QdNcChhu7IBs12IgjXai6WUtq8tFHuS_qFqcyQ0uweisGo-e3WvQ_P1ZPmDeHwMUzHqA4EJEDZPILqrpxblKKFmNM_cw39-vTGAx0lkGdXmYJBb4cXQCYEWrmMCSmsnLVGmEKJiFihvCK3brd-3fWNcwlyReo0d55baa2v8SiPxuKERIdQilfOdVWtLOamT5HLbGs1VKKTXZZXw5F1IoS3OkhmzoQOu9vWX

In [13]:
os.makedirs("share", exist_ok=True)
bucket_id = f"sp-{project_id}-{random_id}"
fetch_config_share(project_id, bucket_id, SOURCE_PATH, CREDENTIALS_PATH)

Valid Delta Sharing configuration found.
config.share er gyldig


In [14]:
# Spesifiser sti til din Delta Sharing config-fil
# Denne filen inneholder credentials og endpoint for din share
CONFIG_FILE = "share/config.share"  # Endre til din config-fil

In [15]:
# Koble til Delta Sharing
sharing_client = delta_sharing.SharingClient(CONFIG_FILE)
tables = sharing_client.list_all_tables()

if not tables:
    raise Exception("❌ Ingen tabeller funnet i sharen")

# Filtrer bort tabeller som ikke er gode for demonstrasjon
# (kode-tabeller, nøkkel-tabeller, krypterte tabeller)
not_valid_table_names = ["kode", "keys", "encrypted"]
valid_examples_tables = [
    table for table in tables
    if not any(substr in table.name for substr in not_valid_table_names)
]

# Velg første egnede tabell (eller første tabell hvis ingen egnede finnes)
table = valid_examples_tables[0] if len(valid_examples_tables) > 0 else tables[0]

# Bygg full tabell-URL for Delta Sharing
table_url = f"{CONFIG_FILE}#{table.share}.{table.schema}.{table.name}"

print(f"✓ Valgt tabell: {table.name}")
print(f"  Share: {table.share}")
print(f"  Schema: {table.schema}")
print(f"  Full URL: {table_url}")

✓ Valgt tabell: dim_kulturminner
  Share: 9ivj-dev
  Schema: matrikkel_silver_v1_ext
  Full URL: share/config.share#9ivj-dev.matrikkel_silver_v1_ext.dim_kulturminner


##  Opprett Spark Session



In [20]:
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F

# Stopp eksisterende Spark-session hvis den finnes
try:
    spark.stop()
    print("🔄 Stoppet eksisterende Spark session")
except:
    pass

# Opprett ny Spark-session med Delta Sharing konfigurasjon
spark = (
    SparkSession.builder
    .appName("DeltaSharingCDF")
    .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:3.1.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)


🔄 Stoppet eksisterende Spark session


# Hent endringer via CDF

1. Beregner et starttidspunkt basert på antall timer tilbake i tid.  
2. Leser endringer fra tabellen siden dette tidspunktet.  
3. Hvis det finnes endringer, grupperes de etter endringstype (`insert`, `update`, `delete`).  
4. Viser en oppsummering av antall endringer per type.  
5. Viser inntil 10 eksempler for hver endringstype.


In [21]:
lookback_hours = 24
start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")

cdf = (
    spark.read.format("deltaSharing")
    .option("responseFormat", "delta")
    .option("readChangeFeed", "true")
    .option("startingTimestamp", start_ts)
    .load(table_url)
)

if not cdf.rdd.isEmpty():
    change_summary = cdf.groupBy("_change_type").count().orderBy("_change_type")
    change_summary.show(truncate=False)

    for row in change_summary.collect():
        change_type = row["_change_type"]
        cdf.filter(F.col("_change_type") == change_type).show(10, truncate=False)


/var/folders/sh/t1kb_fwn67l_f8zrzb26bct40000gn/T/ipykernel_50900/3861382531.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = (datetime.utcnow() - timedelta(hours=lookback_hours)).strftime("%Y-%m-%dT%H:%M:%S.%fZ")
25/10/08 14:27:00 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.RejectedExecutionException: Task java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask@2f663bda[Not completed, task = java.util.concurrent.Executors$RunnableAdapter@259e50aa[Wrapped task = org.apache.spark.rpc.netty.NettyRpcEnv$$anon$1@2b59ba69]] rejected from java.util.concurrent.ScheduledThreadPoolExecutor@6dc6415e[Terminated, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0]
25/10/08 14:27:00 ERROR PythonRunner: Python worker exited unexpectedly (crashed)
org.apache.spark.api.python.PythonExce

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2.0 failed 1 times, most recent failure: Lost task 0.0 in stage 2.0 (TID 3) (10.2.20.233 executor driver): org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.delta.sharing.PreSignedUrlFetcher.getUrl(PreSignedUrlCache.scala:321)
	at io.delta.sharing.client.RandomAccessHttpInputStream.assertNotClosed(RandomAccessHttpInputStream.scala:77)
	at io.delta.sharing.client.RandomAccessHttpInputStream.seek(RandomAcces